# A6 VAE and conditional VAE — core

Adapted from University of Michigan EECS 498/598, Deep Learning for Computer Vision (Winter 2022), by Justin Johnson and course staff.

**Static validation only; not executed.** Enable the run switches explicitly; generated files go to `core_results/`.

In [ ]:
from pathlib import Path
import os
import sys
import torch
import matplotlib.pyplot as plt
candidates = []
for base in (Path.cwd(), *Path.cwd().parents,
             Path('/content/drive/MyDrive/umich-dlcv-wi2022')):
    candidates.extend((base, base / 'assignments' / 'a6'))
ASSIGNMENT_DIR = next((p.resolve() for p in candidates if (p / 'vae_core.py').is_file()), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError('Open from the project/assignment directory, or mount the canonical Colab Drive project first.')
loaded_helpers = sys.modules.get('eecs598')
if loaded_helpers is not None:
    loaded_path = Path(loaded_helpers.__file__).resolve().parent
    if loaded_path != (ASSIGNMENT_DIR / 'eecs598').resolve():
        raise RuntimeError('Restart the kernel before switching assignments: eecs598 is already loaded from another assignment.')
sys.path.insert(0, str(ASSIGNMENT_DIR))
os.chdir(ASSIGNMENT_DIR)
from eecs598.utils import reset_seed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = ASSIGNMENT_DIR / 'core_results' / 'variational_autoencoders'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_OVERFIT = False
ALLOW_DOWNLOAD = False
reset_seed(0)
plt.rcParams['image.cmap'] = 'gray'
print('Outputs:', OUTPUT_DIR)

## MNIST and VAE/CVAE
Training uses the unchanged course helper, which requires CUDA. Both models use latent size 15, batch size 128, Adam at 0.001, and ten epochs.

In [ ]:
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from vae_core import VAE, CVAE
from a6_helper_core import train_vae, show_images
LATENT_SIZE, BATCH_SIZE = 15, 128

In [ ]:
if RUN_TRAINING:
    if DEVICE.type != 'cuda':
        raise RuntimeError('The retained train_vae course helper requires CUDA.')
    mnist = MNIST(str(ASSIGNMENT_DIR / 'MNIST_data'), train=True, download=ALLOW_DOWNLOAD, transform=ToTensor())
    loader_train = torch.utils.data.DataLoader(mnist, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
    for name, cls, conditional in [('vae', VAE, False), ('cvae', CVAE, True)]:
        reset_seed(0)
        model = cls(28 * 28, latent_size=LATENT_SIZE).to(DEVICE)
        for epoch in range(10):
            train_vae(epoch, model, loader_train, cond=conditional)
        torch.save(model.state_dict(), OUTPUT_DIR / f'{name}.pt')

## Samples and latent interpolation
Conditional samples use class vectors for digits 0–9. All figures and checkpoint files are core outputs.

In [ ]:
if RUN_EVALUATION:
    reset_seed(0)
    models = {}
    for name, cls in [('vae', VAE), ('cvae', CVAE)]:
        checkpoint = OUTPUT_DIR / f'{name}.pt'
        if not checkpoint.is_file():
            raise FileNotFoundError(f'Run training first: {checkpoint}')
        models[name] = cls(28 * 28, latent_size=LATENT_SIZE).to(DEVICE)
        models[name].load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True))
        models[name].eval()
    with torch.no_grad():
        z = torch.randn(10, LATENT_SIZE, device=DEVICE)
        samples = models['vae'].decoder(z).cpu()
        labels = torch.eye(10, device=DEVICE)
        conditional_samples = models['cvae'].decoder(torch.cat((torch.randn_like(z), labels), dim=1)).cpu()
    for samples_to_plot, filename in [(samples, 'vae_generation.jpg'), (conditional_samples, 'conditional_vae_generation.jpg')]:
        fig, axes = plt.subplots(1, 10, figsize=(10, 1))
        for ax, sample in zip(axes, samples_to_plot):
            ax.imshow(sample.reshape(28, 28), cmap='Greys_r')
            ax.axis('off')
        fig.savefig(OUTPUT_DIR / filename, bbox_inches='tight')
        plt.show()
    S = 12
    z0 = torch.randn(S, LATENT_SIZE, device=DEVICE)
    z1 = torch.randn(S, LATENT_SIZE, device=DEVICE)
    weights = torch.linspace(0, 1, S, device=DEVICE).view(S, 1, 1)
    z = (weights * z0 + (1 - weights) * z1).transpose(0, 1).reshape(S * S, LATENT_SIZE)
    with torch.no_grad():
        interpolated = models['vae'].decoder(z).cpu()
    show_images(interpolated)
    plt.savefig(OUTPUT_DIR / 'vae_interpolation.jpg', bbox_inches='tight')
    plt.show()